In [1]:
# ==============================================================================
# CELL 1: CÀI ĐẶT MÔI TRƯỜNG
# ==============================================================================
# Cài đặt FinRL và các thư viện hỗ trợ
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git -q
!pip install shimmy>=0.2.1 -q
!pip install pandas numpy matplotlib -q

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# Import các module quan trọng của FinRL
from stable_baselines3 import PPO
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.config import INDICATORS

print("✅ Đã cài đặt xong thư viện.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.7/108.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.9/84.9 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 39.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.0/725.0 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ Đã cài đặt xong thư viện.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
# ==============================================================================
# CELL 2: CLASS MÔI TRƯỜNG VIỆT NAM (ĐÃ FIX LỖI KHỞI TẠO)
# ==============================================================================
class StockTradingEnvVietnam(StockTradingEnv):
    def __init__(self,
                 buy_cost_pct=0.0015,  # Phí mua 0.15%
                 sell_cost_pct=0.0015, # Phí bán 0.15%
                 sell_tax_pct=0.001,   # Thuế bán 0.1% (Luật VN)
                 min_trading_lot=100,  # Lô chẵn 100
                 settlement_cycle=3,   # T+2.5 (Giả lập 3 ngày)
                 **kwargs):

        # --- BƯỚC SỬA LỖI QUAN TRỌNG ---
        # 1. FinRL yêu cầu danh sách 'num_stock_shares' (số cổ phiếu đang có)
        # Nếu chưa có, ta tạo danh sách toàn số 0
        if 'num_stock_shares' not in kwargs:
            stock_dim = kwargs.get('stock_dim')
            kwargs['num_stock_shares'] = [0] * stock_dim

        # 2. Đưa phí mua/bán vào kwargs để gửi cho Class Cha (StockTradingEnv)
        # Vì class Cha bắt buộc phải nhận được 2 tham số này
        kwargs['buy_cost_pct'] = buy_cost_pct
        kwargs['sell_cost_pct'] = sell_cost_pct

        # 3. Gọi khởi tạo của Class Cha
        super().__init__(**kwargs)
        # -------------------------------

        # Cấu hình riêng của Class VN
        self.sell_tax_pct = sell_tax_pct
        self.min_trading_lot = min_trading_lot
        self.settlement_cycle = settlement_cycle

        # Khởi tạo kho Hàng chờ và Tiền chờ
        self.stocks_pending = np.zeros((self.stock_dim, self.settlement_cycle))
        self.cash_pending = np.zeros(self.settlement_cycle)

    def reset(self, **kwargs):
        obs, info = super().reset(**kwargs)
        self.stocks_pending = np.zeros((self.stock_dim, self.settlement_cycle))
        self.cash_pending = np.zeros(self.settlement_cycle)
        return obs, info

    def _update_settlement(self):
        # 1. Tiền về
        cash_arrived = self.cash_pending[-1]
        self.state[0] += cash_arrived
        self.cash_pending = np.roll(self.cash_pending, 1)
        self.cash_pending[0] = 0

        # 2. Cổ phiếu về
        stocks_arrived = self.stocks_pending[:, -1]
        for i in range(self.stock_dim):
            self.state[self.stock_dim + 1 + i] += stocks_arrived[i]
        self.stocks_pending = np.roll(self.stocks_pending, 1, axis=1)
        self.stocks_pending[:, 0] = 0

    def step(self, actions):
        self._update_settlement()
        return super().step(actions)

    def _sell_stock(self, index, action):
        available_stocks = self.state[index + self.stock_dim + 1]

        if available_stocks > 0:
            num_shares = abs(action) // self.min_trading_lot * self.min_trading_lot
            num_shares = min(num_shares, available_stocks)

            if num_shares > 0:
                price = self.state[index + 1]
                total_deduction = self.sell_cost_pct + self.sell_tax_pct
                gross_value = price * num_shares
                net_income = gross_value * (1 - total_deduction)

                self.cash_pending[0] += net_income
                self.state[index + self.stock_dim + 1] -= num_shares
                self.trades += 1
                return num_shares
        return 0

    def _buy_stock(self, index, action):
        num_shares = action // self.min_trading_lot * self.min_trading_lot

        if num_shares > 0:
            price = self.state[index + 1]
            amount_needed = price * num_shares * (1 + self.buy_cost_pct)

            if self.state[0] >= amount_needed:
                self.state[0] -= amount_needed
                self.stocks_pending[index][0] += num_shares
                self.trades += 1
                return num_shares
        return 0

print("✅ Đã sửa lỗi Class StockTradingEnvVietnam thành công.")

✅ Đã sửa lỗi Class StockTradingEnvVietnam thành công.


In [3]:
# ==============================================================================
# CELL 3: LOAD DỮ LIỆU VÀ CẤU HÌNH
# ==============================================================================
# Kiểm tra file
if not os.path.exists("vietnam_vn30_data.csv"):
    print("❌ LỖI: Chưa upload file 'vietnam_vn30_data.csv'.")
else:
    # 1. Load dữ liệu
    df = pd.read_csv("vietnam_vn30_data.csv")
    df['date'] = pd.to_datetime(df['date'])

    # 2. Cắt tập Train (Học từ 2018 đến hết 2023)
    train_df = df[df.date < '2024-01-01'].copy()

    # Reset index cho FinRL
    train_df = train_df.sort_values(['date', 'tic']).reset_index(drop=True)
    train_df.index = train_df.date.factorize()[0]

    print(f"📊 Dữ liệu Train: {len(train_df)} dòng (Kết thúc: 2023-12-31).")

    # 3. Tính toán không gian trạng thái (State Space)
    stock_dimension = len(train_df.tic.unique())
    # Các chỉ báo đã tính ở File 1: macd, rsi_30, cci_30, dx_30
    INDICATORS_LIST = ['macd', 'rsi_30', 'cci_30', 'dx_30']

    state_space = 1 + 2*stock_dimension + len(INDICATORS_LIST)*stock_dimension

    print(f"🔹 Số lượng mã CK: {stock_dimension}")
    print(f"🔹 Kích thước trạng thái (State Space): {state_space}")

📊 Dữ liệu Train: 44940 dòng (Kết thúc: 2023-12-31).
🔹 Số lượng mã CK: 30
🔹 Kích thước trạng thái (State Space): 181


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [8]:
# ==============================================================================
# CELL 4: HUẤN LUYỆN MODEL (TRAINING)
# ==============================================================================
# 1. Cấu hình tham số môi trường
env_kwargs = {
    "hmax": 5000,                 # Mua tối đa 5000 cổ/lệnh
    "initial_amount": 1000000000, # Vốn khởi tạo 1 Tỷ VND
    "reward_scaling": 1e-4,       # Tỷ lệ phần thưởng (giúp AI học ổn định hơn)
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS_LIST,
    "action_space": stock_dimension,

    # --- THAM SỐ VIỆT NAM ---
    "buy_cost_pct": 0.0015,  # Phí mua 0.15%
    "sell_cost_pct": 0.0015, # Phí bán 0.15%
    "sell_tax_pct": 0.001,   # Thuế bán 0.1%
    "min_trading_lot": 100,  # Lô 100
    "settlement_cycle": 3    # T+2.5
}

# 2. Khởi tạo môi trường
e_train_gym = StockTradingEnvVietnam(df=train_df, **env_kwargs)

# 3. Khởi tạo Model PPO
# ent_coef=0.01: Giúp AI chịu khó khám phá các hành động mới
print("🤖 Đang khởi tạo Agent PPO...")
model_ppo = PPO("MlpPolicy", e_train_gym, verbose=1, ent_coef=0.01, learning_rate=0.00025)

# 4. Bắt đầu Train
# Lưu ý: 30,000 bước là con số Demo để chạy nhanh (khoảng 5 phút).
# Để AI thông minh thật sự, bạn nên tăng lên 100,000 hoặc 200,000.
print("⏳ Bắt đầu quá trình học (Training)...")
model_ppo.learn(total_timesteps=30000)

print("🎉 HUẤN LUYỆN HOÀN TẤT!")

# Tạo thư mục chứa model (nếu chưa có)
import os
if not os.path.exists("trained_models"):
    os.makedirs("trained_models")

# Lưu model PPO
save_path_ppo = "trained_models/ppo_vn_agent"
model_ppo.save(save_path_ppo)

print(f"💾 Đã lưu PPO thành công tại: {save_path_ppo}.zip")

🤖 Đang khởi tạo Agent PPO...
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
⏳ Bắt đầu quá trình học (Training)...
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 1.5e+03   |
|    ep_rew_mean     | -2.79e+07 |
| time/              |           |
|    fps             | 114       |
|    iterations      | 1         |
|    time_elapsed    | 17        |
|    total_timesteps | 2048      |
----------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1.5e+03     |
|    ep_rew_mean          | -2.74e+07   |
| time/                   |             |
|    fps                  | 114         |
|    iterations           | 2           |
|    time_elapsed         | 35          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 3.26132e-05 |
|    clip_fraction        | 0           |
|    clip_range           | 0.2         |
|    entropy_loss         | -42.6       |
|    explained_variance   | -4.77e-07   |
|    learning_rate        | 0.00025     |
|    loss                 | 4.83e+10    |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.000611   |
|    std                  | 1           |
|    value_loss           | 9.86e+10    |
-----------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 1.5e+03       |
|    ep_rew_mean          | -2.89e+07     |
| time/                   |               |
|    fps                  | 112           |
|    iterations           | 3             |
|    time_elapsed         | 54            |
|    total_timesteps      | 6144          |
| train/                  |               |
|    approx_kl            | 2.7943315e-05 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -42.6         |
|    explained_variance   | -2.38e-07     |
|    learning_rate        | 0.00025       |
|    loss                 | 4.78e+10      |
|    n_updates            | 20            |
|    policy_gradient_loss | -0.000582     |
|    std                  | 1             |
|    value_loss           | 9.81e+10      |
-------------------------------------------


...

-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 1.5e+03       |
|    ep_rew_mean          | -2.96e+07     |
| time/                   |               |
|    fps                  | 109           |
|    iterations           | 14            |
|    time_elapsed         | 262           |
|    total_timesteps      | 28672         |
| train/                  |               |
|    approx_kl            | 2.1931948e-05 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -42.6         |
|    explained_variance   | 1.19e-07      |
|    learning_rate        | 0.00025       |
|    loss                 | 6.23e+10      |
|    n_updates            | 130           |
|    policy_gradient_loss | -0.000502     |
|    std                  | 1             |
|    value_loss           | 1.21e+11      |
-------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


day: 1497, episode: 20
begin_total_asset: 1000000000.00
end_total_asset: 471987115.99
total_reward: -528012884.01
total_cost: 0.00
total_trades: 9571
Sharpe: 0.710


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 1.5e+03       |
|    ep_rew_mean          | -2.98e+07     |
| time/                   |               |
|    fps                  | 109           |
|    iterations           | 15            |
|    time_elapsed         | 281           |
|    total_timesteps      | 30720         |
| train/                  |               |
|    approx_kl            | 1.9454543e-05 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -42.6         |
|    explained_variance   | 0             |
|    learning_rate        | 0.00025       |
|    loss                 | 5.46e+10      |
|    n_updates            | 140           |
|    policy_gradient_loss | -0.000448     |
|    std                  | 1             |
|    value_loss           | 1.07e+11      |
-------------------------------------------
🎉 HUẤN LUYỆN HOÀN TẤT!
💾 Đã lưu 

In [7]:
# ==============================================================================
# CELL 5: HUẤN LUYỆN MODEL A2C
# ==============================================================================
from stable_baselines3 import A2C

print("🤖 [2/3] Đang khởi tạo Agent A2C...")

# A2C dùng chung môi trường e_train_gym mà ta đã tạo ở trên
model_a2c = A2C("MlpPolicy",
                e_train_gym,
                verbose=1,
                ent_coef=0.01,
                learning_rate=0.0005)

print("⏳ Bắt đầu Training A2C (khoảng 3-5 phút)...")
model_a2c.learn(total_timesteps=30000)

# Lưu model
save_path_a2c = "trained_models/a2c_vn_agent"
model_a2c.save(save_path_a2c)

print(f"✅ Đã lưu model A2C tại: {save_path_a2c}.zip")

🤖 [2/3] Đang khởi tạo Agent A2C...
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
⏳ Bắt đầu Training A2C (khoảng 3-5 phút)...
-------------------------------------
| time/                 |           |
|    fps                | 101       |
|    iterations         | 100       |
|    time_elapsed       | 4         |
|    total_timesteps    | 500       |
| train/                |           |
|    entropy_loss       | -42.7     |
|    explained_variance | -2.84e-05 |
|    learning_rate      | 0.0005    |
|    n_updates          | 99        |
|    policy_loss        | -9.75e+05 |
|    std                | 1.01      |
|    value_loss         | 7.23e+08  |
-------------------------------------
------------------------------------
| time/                 |          |
|    fps                | 97       |
|    iterations         | 200      |
|    time_elapsed       | 10       |
|    total_timesteps    | 1000     |
| train/                |          

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1.5e+03   |
|    ep_rew_mean        | -1.67e+07 |
| time/                 |           |
|    fps                | 103       |
|    iterations         | 400       |
|    time_elapsed       | 19        |
|    total_timesteps    | 2000      |
| train/                |           |
|    entropy_loss       | -42.8     |
|    explained_variance | 2.09e-06  |
|    learning_rate      | 0.0005    |
|    n_updates          | 399       |
|    policy_loss        | -4.01e+05 |
|    std                | 1.01      |
|    value_loss         | 1.07e+08  |
-------------------------------------
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1.5e+03   |
|    ep_rew_mean        | -1.67e+07 |
| time/                 |           |
|    fps                | 101       |
|    iterations         | 500       |
|    time_elapsed       | 24        |
|    total_t

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1.5e+03   |
|    ep_rew_mean        | -1.15e+07 |
| time/                 |           |
|    fps                | 101       |
|    iterations         | 700       |
|    time_elapsed       | 34        |
|    total_timesteps    | 3500      |
| train/                |           |
|    entropy_loss       | -43       |
|    explained_variance | 0         |
|    learning_rate      | 0.0005    |
|    n_updates          | 699       |
|    policy_loss        | -2.04e+05 |
|    std                | 1.01      |
|    value_loss         | 3.32e+07  |
-------------------------------------
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1.5e+03   |
|    ep_rew_mean        | -1.15e+07 |
| time/                 |           |
|    fps                | 102       |
|    iterations         | 800       |
|    time_elapsed       | 38        |
|    total_t

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1.5e+03   |
|    ep_rew_mean        | -2.28e+06 |
| time/                 |           |
|    fps                | 105       |
|    iterations         | 5500      |
|    time_elapsed       | 261       |
|    total_timesteps    | 27500     |
| train/                |           |
|    entropy_loss       | -43.1     |
|    explained_variance | 0         |
|    learning_rate      | 0.0005    |
|    n_updates          | 5499      |
|    policy_loss        | -5.98e+04 |
|    std                | 1.02      |
|    value_loss         | 2.92e+06  |
-------------------------------------
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1.5e+03   |
|    ep_rew_mean        | -2.28e+06 |
| time/                 |           |
|    fps                | 105       |
|    iterations         | 5600      |
|    time_elapsed       | 266       |
|    total_t

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1.5e+03   |
|    ep_rew_mean        | -2.19e+06 |
| time/                 |           |
|    fps                | 105       |
|    iterations         | 5800      |
|    time_elapsed       | 275       |
|    total_timesteps    | 29000     |
| train/                |           |
|    entropy_loss       | -43.1     |
|    explained_variance | 1.79e-07  |
|    learning_rate      | 0.0005    |
|    n_updates          | 5799      |
|    policy_loss        | -2.42e+04 |
|    std                | 1.02      |
|    value_loss         | 9.89e+05  |
-------------------------------------
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1.5e+03   |
|    ep_rew_mean        | -2.19e+06 |
| time/                 |           |
|    fps                | 105       |
|    iterations         | 5900      |
|    time_elapsed       | 280       |
|    total_t

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/save_util.py:284: UserWarning: Path 'trained_models' does not exist. Will create it.
  warnings.warn(f"Path '{path.parent}' does not exist. Will create it.")


In [9]:
# ==============================================================================
# CELL 6: HUẤN LUYỆN MODEL DDPG
# ==============================================================================
from stable_baselines3 import DDPG
from stable_baselines3.common.noise import NormalActionNoise

print("🤖 [3/3] Đang khởi tạo Agent DDPG...")

# 1. Tạo Nhiễu (Noise) để AI chịu khó khám phá
# (DDPG là thuật toán off-policy nên cần cái này)
n_actions = e_train_gym.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

# 2. Khởi tạo Model
model_ddpg = DDPG("MlpPolicy",
                  e_train_gym,
                  action_noise=action_noise,
                  verbose=1,
                  learning_rate=0.0005)

print("⏳ Bắt đầu Training DDPG (Model này chạy chậm hơn PPO/A2C một chút)...")
model_ddpg.learn(total_timesteps=30000)

# 3. Lưu Model
save_path_ddpg = "trained_models/ddpg_vn_agent"
model_ddpg.save(save_path_ddpg)

print(f"✅ Đã lưu model DDPG tại: {save_path_ddpg}.zip")

🤖 [3/3] Đang khởi tạo Agent DDPG...
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
⏳ Bắt đầu Training DDPG (Model này chạy chậm hơn PPO/A2C một chút)...
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 1.5e+03   |
|    ep_rew_mean     | -5.93e+05 |
| time/              |           |
|    episodes        | 4         |
|    fps             | 20        |
|    time_elapsed    | 286       |
|    total_timesteps | 5992      |
| train/             |           |
|    actor_loss      | 5.31e+04  |
|    critic_loss     | 5.17e+07  |
|    learning_rate   | 0.0005    |
|    n_updates       | 5891      |
----------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/           |           |
|    ep_len_mean     | 1.5e+03   |
|    ep_rew_mean     | -3.37e+05 |
| time/              |           |
|    episodes        | 8         |
|    fps             | 20        |
|    time_elapsed    | 575       |
|    total_timesteps | 11984     |
| train/             |           |
|    actor_loss      | 3.72e+04  |
|    critic_loss     | 2.85e+07  |
|    learning_rate   | 0.0005    |
|    n_updates       | 11883     |
----------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


day: 1497, episode: 30
begin_total_asset: 1000000000.00
end_total_asset: 1643316614.40
total_reward: 643316614.40
total_cost: 0.00
total_trades: 9
Sharpe: 0.408


...

day: 1497, episode: 40
begin_total_asset: 1000000000.00
end_total_asset: 1490164353.19
total_reward: 490164353.19
total_cost: 0.00
total_trades: 10
Sharpe: 0.406


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/           |           |
|    ep_len_mean     | 1.5e+03   |
|    ep_rew_mean     | -1.71e+05 |
| time/              |           |
|    episodes        | 20        |
|    fps             | 20        |
|    time_elapsed    | 1457      |
|    total_timesteps | 29960     |
| train/             |           |
|    actor_loss      | 1.9e+04   |
|    critic_loss     | 4.49e+06  |
|    learning_rate   | 0.0005    |
|    n_updates       | 29859     |
----------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ Đã lưu model DDPG tại: trained_models/ddpg_vn_agent.zip
